# 03 校验、修订与查询更新

目标：区分“记录格式合理”和“事实符合原文”，保存每次修订及依据，并观察查询结果随修订变化。

本 Notebook 显式分步执行，不调用一键 `demo`。请先完成 01、02，使用同一个实验目录。SQLite 当前记录、历史和修订协议属于课程代码；ORD 提供的是其真实记录格式与验证器。

中途会**故意把产率从 72% 改成 27%**，用于演示“数值合法但事实错误”。这是明确标注的教学注入，不是 ReactionSeek 的实际错误，也不是文献报告值。

In [ ]:
from pathlib import Path
import os
import sys
import json

# 从仓库根目录或 notebooks/ 打开均可；不依赖个人机器的绝对路径。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/reaction_knowledge_lab").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请在 reaction-knowledge-lab 仓库根目录或 notebooks 目录启动。")
sys.path.insert(0, str(ROOT / "src"))
run_setting = Path(os.environ.get("RKL_NOTEBOOK_RUN_DIR", "runs/notebook-lab")).expanduser()
LAB = run_setting.resolve() if run_setting.is_absolute() else (ROOT / run_setting).resolve()
LAB.mkdir(parents=True, exist_ok=True)
RAW, ALIGNED, DB = LAB / "extracted.csv", LAB / "aligned.json", LAB / "knowledge.sqlite"
from reaction_knowledge_lab.files import DATA, read_json, write_json, load_records, save_records
print("仓库：", ROOT)
print("本次实验目录：", LAB)
print("Python：", sys.executable)

In [ ]:
if not ALIGNED.is_file():
    raise FileNotFoundError(f"请先完成 02，预期文件：{ALIGNED}")
if DB.exists():
    raise FileExistsError(f"数据库已存在，拒绝覆盖：{DB}。请换新实验目录从 01 重做。")
from reaction_knowledge_lab.alignment import entity_cache
from reaction_knowledge_lab.store import build_database, query_records, get_records, get_history, apply_patch
from reaction_knowledge_lab.workflow import (
    FIRST, SECOND, teaching_error_patch, reference_patch, validate_and_export,
)
from reaction_knowledge_lab.validation import check_evidence
records = load_records(ALIGNED)

## 1. 同时看原文参考核对与 ORD 报告

原文参考核对会提示漏掉第二次实验、第一条记录缺少纯度。ORD 检查输入、产物、单位等结构约束，不能自行知道文献究竟报告了什么。

In [ ]:
initial_report = validate_and_export(records, LAB / "01_initial")
print("原文参考差异：")
for issue in initial_report["source_check"]["issues"]:
    print(issue["code"], issue.get("record_id"), issue.get("field"),
          "实际=", issue.get("actual"), "参考=", issue.get("expected"))
print("ORD结构合格记录：", initial_report["ord_structurally_ready_ids"])
assert any(i["code"] == "missing_experiment" and i["record_id"] == SECOND
           for i in initial_report["source_check"]["issues"])
assert FIRST in initial_report["ord_structurally_ready_ids"]

## 2. 建立知识库并固定一条查询

下面创建新数据库。数据库分别保存实体、实验记录、关联和修订历史；已有数据库不会覆盖。固定查询为“产率至少 70% 的实验”。初始应返回 **1 条**。

In [ ]:
print(build_database(records, DB, entity_cache()))
query_initial = query_records(DB, min_yield=70)
write_json(LAB / "query_initial.json", query_initial)
print([(r["record_id"], r["product"]["yield_percent"], r["revision"]) for r in query_initial])
assert len(query_initial) == 1

## 3. 先审阅教学错误提案，再应用

提案文件有独立 patch ID、预期版本、理由和教学标记。它记录“改什么”，不静默覆盖整份知识库。

In [ ]:
injected_patch = teaching_error_patch(DB)
write_json(LAB / "teaching_error.patch.json", injected_patch)
print(json.dumps(injected_patch, ensure_ascii=False, indent=2))
assert injected_patch["teaching_injection"] is True
assert injected_patch["changes"][0]["set"]["product.yield_percent"] == 27

In [ ]:
print(apply_patch(DB, injected_patch))
injected_report = validate_and_export(get_records(DB), LAB / "02_injected_error")
query_injected = query_records(DB, min_yield=70)
write_json(LAB / "query_injected.json", query_injected)
yield_errors = [i for i in injected_report["source_check"]["issues"]
                if i["code"] == "source_mismatch" and i.get("field") == "product.yield_percent"]
print("产率原文核对结果：", yield_errors)
print("ORD结构合格记录：", injected_report["ord_structurally_ready_ids"])
print("产率至少70%的查询条数：", len(query_injected))
assert yield_errors and FIRST in injected_report["ord_structurally_ready_ids"]
assert len(query_injected) == 0

27% 落在合法百分比范围内，因此通过 ORD 结构规则不代表它符合原文。这里靠课程参考定位错误；任意新文献仍需要来源核对，不能只依赖格式验证或模型合理性分数。

## 4. 根据原文参考生成修订提案

本步骤使用已给出的课程参考，是**参考辅助修订**，不是模型自动阅读文献并纠错。提案将第一条记录修订为来源所述内容，并补录第二次实验；原始抽取 `raw` 仍被保留。

请先看动作摘要，并打开实验目录中的 `source_review.patch.json` 核对具体字段及证据，再运行下一格应用。

In [ ]:
correction = reference_patch(DB)
write_json(LAB / "source_review.patch.json", correction)
print("修订理由：", correction["reason"])
print("参考文件：", correction["reference_file"])
print("证据条数：", len(correction["evidence"]))
for change in correction["changes"]:
    if change["action"] == "add":
        print("新增：", change["record"]["record_id"], "产物质量：", change["record"]["product"]["mass_g"])
    else:
        print("替换字段：", change["record_id"], sorted(change["set"]))
print("完整提案：", LAB / "source_review.patch.json")
assert any(c["action"] == "add" and c["record"]["record_id"] == SECOND for c in correction["changes"])

In [ ]:
print(apply_patch(DB, correction))
final_records = get_records(DB)
save_records(LAB / "final_records.json", final_records)
query_reviewed = query_records(DB, min_yield=70)
write_json(LAB / "query_reviewed.json", query_reviewed)
print([(r["record_id"], r["product"]["mass_g"], r["product"]["yield_percent"], r["revision"])
       for r in query_reviewed])
assert [len(query_initial), len(query_injected), len(query_reviewed)] == [1, 0, 2]
assert len({r["product"]["entity_id"] for r in final_records}) == 1
assert sorted(r["product"]["mass_g"] for r in final_records) == [2.62, 2.63]
print("按产物别名查询：", [r["record_id"] for r in query_records(DB, name="diphenylmethanone")])
assert len(query_records(DB, name="diphenylmethanone")) == 2

## 5. 来源一致，但记录仍可能不完整

第二次实验明确报告了产物质量、产率和纯度，但相应段落没有逐项重述投料与条件。本课程保留这些缺失，不复制第一条条件来制造一条完整训练样本。

因此最终应出现：两条记录都符合这份课程参考，只有第一条通过当前 ORD 结构检查，第二条仍标记为不完整。请分别查看报告中的 errors 和 warnings。

In [ ]:
reviewed_report = validate_and_export(final_records, LAB / "03_reviewed")
print("与课程参考一致：", reviewed_report["source_check"]["source_reference_agreement"])
print("ORD结构合格：", reviewed_report["ord_structurally_ready_ids"])
print("ORD不完整：", reviewed_report["ord_incomplete_ids"])
r2 = next(r for r in final_records if r["record_id"] == SECOND)
print("第二次实验保留的条件：", r2["conditions"])
print("第二次实验ORD错误：", next(r["errors"] for r in reviewed_report["ord"]["records"] if r["record_id"] == SECOND))
assert reviewed_report["source_check"]["source_reference_agreement"] is True
assert reviewed_report["ord_structurally_ready_ids"] == [FIRST]
assert reviewed_report["ord_incomplete_ids"] == [SECOND]
assert r2["conditions"]["time_minutes"] is None and r2["conditions"]["temperature_c"] is None
assert r2["inputs"] == []

## 6. 查看历史，并检查重复提交不会产生新版本

第一条记录应经历 72 → 27 → 72，三次状态都有记录，原始抽取仍保留 72%。重复应用完全相同的修订提案应返回 `already_applied`，不会再创建历史版本。

In [ ]:
history = get_history(DB, FIRST)
write_json(LAB / "history_run1.json", history)
for event in history:
    print(event["revision"], event["record"]["product"]["yield_percent"], event["reason"])
assert [event["record"]["product"]["yield_percent"] for event in history] == [72, 27, 72]
assert next(r for r in final_records if r["record_id"] == FIRST)["raw"]["Yield"] == "72%"
repeat = apply_patch(DB, correction)
print("重复应用：", repeat)
assert repeat["status"] == "already_applied"
assert len(get_history(DB, FIRST)) == len(history)

## 7. 保存验收结论

以下结论针对这个小案例，不代表论文全数据指标，也不代表已经获得两条完整训练样本。原文位置检查若返回 `not_checked`，需要按主手册下载原文后再做核对。

In [ ]:
evidence_status = check_evidence()
write_json(LAB / "evidence_check.json", evidence_status)
checks = {
    "query_counts_1_0_2": [len(query_initial), len(query_injected), len(query_reviewed)] == [1, 0, 2],
    "two_independent_experiments": {r["record_id"] for r in final_records} == {FIRST, SECOND},
    "shared_product_entity": len({r["product"]["entity_id"] for r in final_records}) == 1,
    "source_reference_agreement": reviewed_report["source_check"]["source_reference_agreement"],
    "r2_remains_incomplete": reviewed_report["ord_incomplete_ids"] == [SECOND],
    "history_72_27_72": [event["record"]["product"]["yield_percent"] for event in history] == [72, 27, 72],
    "patch_replay_is_idempotent": repeat["status"] == "already_applied" and len(get_history(DB, FIRST)) == len(history),
    "no_api_calls_in_replay": read_json(RAW.with_suffix(".metadata.json"))["api_calls"] == 0,
}
assert all(checks.values()), checks
verification = {
    "passed": all(checks.values()), "checks": checks, "api_calls": 0,
    "query_counts_initial_injected_reviewed": [len(query_initial), len(query_injected), len(query_reviewed)],
    "record_ids": [r["record_id"] for r in final_records],
    "shared_product_entity": final_records[0]["product"]["entity_id"],
    "source_reference_agreement": reviewed_report["source_check"]["source_reference_agreement"],
    "ord_structurally_ready_ids": reviewed_report["ord_structurally_ready_ids"],
    "ord_incomplete_ids": reviewed_report["ord_incomplete_ids"],
    "history_yields": [event["record"]["product"]["yield_percent"] for event in history],
    "patch_replay_status": repeat["status"], "evidence_check": evidence_status,
}
write_json(LAB / "notebook_verification.json", verification)
print(json.dumps(verification, ensure_ascii=False, indent=2))

**提交内容：** `knowledge.sqlite`、三个阶段的 `validation.json`、修订提案、三份查询结果、`history_run1.json`、`final_records.json` 和 `notebook_verification.json`。

回答：

1. 为什么 27% 能通过 ORD 检查，却不符合原文？
2. 为什么第二次实验可以进入可追溯知识库，却仍不能直接当完整训练样本？
3. 为什么应保留修改历史，而不是直接用最终值覆盖最初结果？

如需进一步实验，可阅读 `optional/rxn_filter/README.md`；模型结构合理性检查不会替代这里的来源与版本检查。